# 13-phase3-seed-robustness

**neuron Phase 3** — Phase 2 의 sweet spot (α_init=0.10) 시드 robustness 검증 + GraphLM Phase 1 직접 비교.

- **Phase 2 의 발견** (PR #44): α_init=0.10 에서 dead **0%**, final_loss **1.7747** (baseline 1.7865 대비 0.0118 개선). 단일 시드 (42) 결과.
- **본 Phase 의 가설**:
  1. **시드 invariance** — α_init=0.10 의 dead 0% / loss 개선이 다른 시드 (123, 456) 에서도 재현?
  2. **분산** — 시드 간 final_loss σ 가 합리적 범위 (< 0.02)?
  3. **GraphLM Phase 1 직접 비교** — neuron Phase 2 의 final_loss 가 GraphLM (4-layer 1.7871) 보다 일관 우위?
- **설계**: 2 × 3 sweep (α_init ∈ {0.0, 0.10} × seed ∈ {42, 123, 456}) = 6 run, GPU 약 9분.
- **작성일**: 2026-05-25
- **연관**: Issue [#45](https://github.com/EinSofINTEREST/GraphLM/issues/45) / Phase 2 baseline PR [#44](https://github.com/EinSofINTEREST/GraphLM/pull/44)

## 1. 환경 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import NeuronConfig, NeuronGrowingDecoder, add_attn_smooth_start
from graphlm.training.triggers import PlateauTrigger
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)

print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)
if str(DEVICE).startswith("cuda"):
    print(f"  device 0      : {torch.cuda.get_device_name(0)}")

## 2. 실험 설정 — 2 × 3 sweep

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]
ALPHA_INITS = [0.0, 0.10]  # baseline (Phase 1/2 와 동등) + sweet spot

BATCH_SIZE = 16
BLOCK_SIZE = 128
BACKBONE = dict(hidden_dim=256, n_heads=4, ffn_dim=1024, n_layers=4, n_init_attn=1)
TRAIN = dict(
    lr=3e-4,
    max_steps=1500,
    max_total_attn=8,
    trigger_window=100,
    trigger_epsilon=0.04,
    trigger_cooldown=150,
    trigger_min_history=100,
)
DEAD_THRESHOLD = 0.05

# GraphLM Phase 1 baseline (PR #38 의 4-layer NO_FIRE 결과)
GRAPHLM_PHASE1_4L_FINAL_LOSS = 1.7871

print(f"SEEDS       = {SEEDS}")
print(f"ALPHA_INITS = {ALPHA_INITS}")
print(f"총 run      = {len(SEEDS) * len(ALPHA_INITS)} (GPU 약 9분 예상)")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
print(f"vocab_size : {tokenizer.vocab_size}, n_tokens : {len(dataset):,}")

## 4. 2 × 3 sweep 학습

각 (α_init, seed) 조합마다 fresh model + train 1500 step + round-robin attn add.

In [ ]:
cfg = NeuronConfig(vocab_size=tokenizer.vocab_size, max_seq_len=BLOCK_SIZE, **BACKBONE)

# {(alpha_init, seed): {'losses': [...], 'grow_events': [...], 'final_alphas': [...]}}
runs = {}
for alpha_init in ALPHA_INITS:
    for seed in SEEDS:
        print(f"--- alpha_init={alpha_init}, seed={seed} ---")
        set_seed(seed)
        model = NeuronGrowingDecoder(cfg).to(DEVICE)
        data_iter = iter_random_batches(
            dataset, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE, seed=seed
        )
        trigger = PlateauTrigger(
            window=TRAIN["trigger_window"],
            epsilon=TRAIN["trigger_epsilon"],
            cooldown=TRAIN["trigger_cooldown"],
            min_history=TRAIN["trigger_min_history"],
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=TRAIN["lr"])

        losses = []
        grow_events = []
        next_block_to_grow = 0
        V = cfg.vocab_size
        model.train()
        for step in range(1, TRAIN["max_steps"] + 1):
            x, y = next(data_iter)
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            if (
                trigger.update(loss.item())
                and sum(model.n_attn_per_block) < TRAIN["max_total_attn"]
            ):
                target_block = next_block_to_grow
                add_attn_smooth_start(model, target_block, alpha_init=alpha_init)
                block = model.blocks[target_block]
                new_params = (
                    list(block.attn_lns[-1].parameters())
                    + list(block.attns[-1].parameters())
                    + [block.attn_alphas[-1]]
                )
                # lr 명시 (PR #44 의 fix 적용)
                optimizer.add_param_group({"params": new_params, "lr": TRAIN["lr"]})
                grow_events.append((step, target_block, sum(model.n_attn_per_block)))
                next_block_to_grow = (next_block_to_grow + 1) % cfg.n_layers

        final_alphas = [
            (bi, ai, alpha_param.item())
            for bi, block in enumerate(model.blocks)
            for ai, alpha_param in enumerate(block.attn_alphas)
        ]
        runs[(alpha_init, seed)] = {
            "losses": losses,
            "grow_events": grow_events,
            "final_alphas": final_alphas,
        }
        n_last = min(100, len(losses))
        final_loss = sum(losses[-n_last:]) / n_last if n_last > 0 else 0.0
        print(f"  done: final_loss(last 100 avg)={final_loss:.4f}")

## 5. 결과 표 — 시드 × α_init

In [ ]:
import statistics

print(f"{'α_init':>7}  {'seed':>5}  {'dead%':>6}  {'mean|α_add|':>11}  {'final_loss':>11}")
print("-" * 55)
table = {}  # {alpha_init: {seed: {'dead_pct': ..., 'mean_add': ..., 'final_loss': ...}}}
for alpha_init in ALPHA_INITS:
    table[alpha_init] = {}
    for seed in SEEDS:
        r = runs[(alpha_init, seed)]
        added = [(bi, ai, a) for bi, ai, a in r["final_alphas"] if ai >= BACKBONE["n_init_attn"]]
        added_abs = [abs(a) for _, _, a in added]
        dead_pct = (
            sum(1 for v in added_abs if v < DEAD_THRESHOLD) / len(added_abs) if added_abs else 0.0
        )
        mean_add = statistics.mean(added_abs) if added_abs else 0.0
        n_last = min(100, len(r["losses"]))
        final_loss = sum(r["losses"][-n_last:]) / n_last if n_last > 0 else 0.0
        table[alpha_init][seed] = dict(dead_pct=dead_pct, mean_add=mean_add, final_loss=final_loss)
        print(
            f"{alpha_init:>7.2f}  {seed:>5}  {dead_pct:>6.1%}  {mean_add:>11.4f}  {final_loss:>11.4f}"
        )

## 6. α_init 별 시드 invariance 분석

In [ ]:
print(f"{'α_init':>7}  {'dead% range':>15}  {'final_loss mean':>15}  {'σ':>7}  verdict")
print("-" * 75)
for alpha_init in ALPHA_INITS:
    dead_pcts = [table[alpha_init][s]["dead_pct"] for s in SEEDS]
    final_losses = [table[alpha_init][s]["final_loss"] for s in SEEDS]
    dead_range = f"{min(dead_pcts):.0%} - {max(dead_pcts):.0%}"
    fl_mean = statistics.mean(final_losses)
    fl_sigma = statistics.stdev(final_losses) if len(final_losses) > 1 else 0
    if alpha_init == 0.0:
        verdict = "baseline (Phase 1/2 와 일관 예상)"
    elif max(dead_pcts) == 0:
        verdict = "✅ 모든 시드 dead 회피 (robust)"
    elif min(dead_pcts) == 1.0:
        verdict = "❌ 모든 시드 dead (회피 실패)"
    else:
        verdict = "⚠️ 일부 시드 dead (시드 의존)"
    print(f"{alpha_init:>7.2f}  {dead_range:>15}  {fl_mean:>15.4f}  {fl_sigma:>7.4f}  {verdict}")

## 7. GraphLM Phase 1 직접 비교

neuron (head-level) α_init=0.10 vs GraphLM (block-level) 4-layer baseline (1.7871).

In [ ]:
neuron_010_losses = [table[0.10][s]["final_loss"] for s in SEEDS]
neuron_010_mean = statistics.mean(neuron_010_losses)
neuron_010_sigma = statistics.stdev(neuron_010_losses) if len(neuron_010_losses) > 1 else 0

print("neuron Phase 2/3 (α_init=0.10, head-level):")
print(f"  seeds {SEEDS}: {[round(v, 4) for v in neuron_010_losses]}")
print(f"  mean ± σ : {neuron_010_mean:.4f} ± {neuron_010_sigma:.4f}")
print()
print("GraphLM Phase 1 (block-level, 4-layer baseline, PR #38):")
print(f"  final_loss : {GRAPHLM_PHASE1_4L_FINAL_LOSS:.4f}")
print()
diff = GRAPHLM_PHASE1_4L_FINAL_LOSS - neuron_010_mean
print(
    f"neuron vs GraphLM 차이 : {diff:+.4f}  ({'neuron 우위' if diff > 0 else 'GraphLM 우위' if diff < 0 else 'tie'})"
)
if diff > 0:
    n_better = sum(1 for v in neuron_010_losses if v < GRAPHLM_PHASE1_4L_FINAL_LOSS)
    print(f"  → 3 시드 중 {n_better} 시드에서 neuron 이 GraphLM 보다 낮은 loss")

## 8. 시각화 — 시드별 final_loss 그룹 bar

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# (a) 시드별 final_loss bar (α_init grouped)
width = 0.35
x_seeds = list(range(len(SEEDS)))
for i, alpha_init in enumerate(ALPHA_INITS):
    offsets = [x + (i - 0.5) * width for x in x_seeds]
    vals = [table[alpha_init][s]["final_loss"] for s in SEEDS]
    ax1.bar(offsets, vals, width=width, label=f"α_init={alpha_init}", alpha=0.85)
ax1.axhline(
    GRAPHLM_PHASE1_4L_FINAL_LOSS, color="red", linestyle="--", lw=1, label="GraphLM 4L baseline"
)
ax1.set_xlabel("seed")
ax1.set_xticks(x_seeds)
ax1.set_xticklabels([str(s) for s in SEEDS])
ax1.set_ylabel("final loss (last 100 avg)")
ax1.set_title("final loss: α_init × seed + GraphLM baseline")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(alpha=0.3, axis="y")
ax1.set_ylim(min(GRAPHLM_PHASE1_4L_FINAL_LOSS, min(neuron_010_losses)) - 0.01, 1.81)

# (b) 시드별 dead %
for i, alpha_init in enumerate(ALPHA_INITS):
    offsets = [x + (i - 0.5) * width for x in x_seeds]
    vals = [table[alpha_init][s]["dead_pct"] * 100 for s in SEEDS]
    ax2.bar(offsets, vals, width=width, label=f"α_init={alpha_init}", alpha=0.85)
ax2.set_xlabel("seed")
ax2.set_xticks(x_seeds)
ax2.set_xticklabels([str(s) for s in SEEDS])
ax2.set_ylabel("dead %")
ax2.set_title("dead %: α_init × seed")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(alpha=0.3, axis="y")
ax2.set_ylim(0, 110)

fig.tight_layout()
fig.savefig(OUT_DIR / "seed_robustness.png", dpi=120)
plt.show()

## 결과 요약 / Phase 4 권장 방향

이 노트북에서 확인할 것:
- §6 의 α_init=0.10 verdict — "모든 시드 dead 회피" 또는 "일부 시드 dead"
- §7 의 GraphLM 직접 비교 — neuron 이 3 시드 모두에서 GraphLM 보다 낮은 loss?
- §6 의 final_loss σ — 시드 간 분산이 합리적인가 (< 0.02)?

**판정 시나리오**:
- **A. α_init=0.10 모든 시드 robust + GraphLM 우위** ⭐
  - sweet spot 의 robustness 확정 → 더 세밀한 α_init sweep (Phase 4)
  - 또는 옵션 A (connection-level α) 로 직접 확장
- **B. 일부 시드 dead** — sweet spot 이 시드 의존
  - seed × α_init 2D sweep 으로 정밀 탐색
  - α_init scheduler (warmup-style) 또는 noise 추가
- **C. GraphLM 와 비교 시 비슷/열위**
  - 비교 자체가 fair 한지 재점검 (parameter 수 / 학습 시간 등)
  - 또는 task 가 너무 작아 차이가 드러나지 않음 → 더 큰 task / 긴 학습 검토

**가장 야심찬 시나리오 (A 확정 시)**:
- neuron Phase 4 — 옵션 A (connection-level α) 로 expand. α 가 matrix 면 자연스럽게 α_init 도 fine-grained 분포 가능.
- neuron Phase 5 — 옵션 C (local activation/gradient trigger) hybrid.